# 01 · Main introspection sweep

Hours 9-17. Conditions C1-C4 at every lambda, structured prompt variant.

Every generation is appended to JSONL as it is produced and keyed by
`(lam, condition, concept, trial, variant)`, so re-running this cell after a
session kill resumes exactly where it stopped.

In [ ]:
#@title Clone the repo and install dependencies { display-mode: "form" }
# Colab: paste a GitHub PAT with repo:read scope. It is used only for the clone
# and is not written to disk.
import os, subprocess, sys, getpass, pathlib

REPO   = "sagnikc395/apart-mind-digital-mind"  #@param {type:"string"}
BRANCH = "main"                                 #@param {type:"string"}
WORKDIR = "/content"

if pathlib.Path("/content").exists():
    token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub token (blank if public): ")
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    dest = pathlib.Path(WORKDIR) / REPO.split("/")[-1]
    if dest.exists():
        subprocess.run(["git", "-C", str(dest), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", url, str(dest)], check=True)
    os.chdir(dest)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "torch", "transformers>=4.44", "accelerate", "datasets", "matplotlib"], check=True)
else:
    os.chdir(pathlib.Path.cwd())  # already inside the repo, e.g. running locally

sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
print("cwd:", os.getcwd())


In [ ]:
#@title Persist results to Drive (survives a session kill)
import os, pathlib

RESULTS = "/content/drive/MyDrive/alignment_tax_results"  #@param {type:"string"}
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    RESULTS = str(pathlib.Path.cwd() / "results")
    print("no Drive; writing to", RESULTS, f"({exc})")
os.environ["ALIGNMENT_TAX_RESULTS"] = RESULTS
pathlib.Path(RESULTS).mkdir(parents=True, exist_ok=True)
print("results ->", RESULTS)


In [ ]:
#@title Reload the locked configuration from the pilot
from pathlib import Path
import os
from alignment_tax.config import RunConfig
from alignment_tax import pipeline

results = Path(os.environ.get("ALIGNMENT_TAX_RESULTS", "results"))
cfg = RunConfig.load(next(results.rglob("run_config.json")))
cfg.results_dir = results
print("layer:", cfg.injection.layer, "alpha:", cfg.injection.alpha, "lambdas:", cfg.lambdas)

hm = pipeline.load_model(cfg)
rd = pipeline.stage_direction(hm, cfg)
bank = pipeline.stage_concepts(hm, cfg)

In [ ]:
#@title Main sweep (resumable -- just re-run this cell if the session dies)
path = pipeline.stage_sweep(hm, cfg, bank, rd.vector, variant="structured")
print(path)

In [ ]:
#@title Robustness: skeptical prompt variant at the lambda endpoints only
pipeline.stage_sweep(hm, cfg, bank, rd.vector, variant="skeptical",
                     lambdas=(min(cfg.lambdas), max(cfg.lambdas)))

In [ ]:
#@title Quick look at where things stand
from alignment_tax.stats import load_and_summarise
import pandas as pd

records, summary = load_and_summarise(cfg.artifact("sweep_structured.jsonl"), n_boot=500)
pd.DataFrame([{"lam": lam, **{k: round(v.value, 3) for k, v in m.items()}} for lam, m in summary.items()])